# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Instantiate the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Get the metadata object (not as dict)
metadata = dataset.metadata

print(f"\033[1m{metadata.name}\033[0m\n{metadata.description}")

## 2. Data Overview

Review available record sets, their `@id`s, and examine their fields/columns.

Entities (such as record sets and fields) are always referenced **by their `@id`** for consistency and reproducibility.

In [ ]:
# List available record sets and fields using the Croissant schema

print('Available record sets:')
rs_list = []
for record_set in metadata.record_sets:
    print(f"  - name: {record_set.name}, @id: {record_set.id}")
    rs_list.append(record_set.id)
    if hasattr(record_set, 'fields') and record_set.fields:
        print("    Fields:")
        for field in record_set.fields:
            print(f"      - {field.name} (@id: {field.id}, type: {getattr(field, 'data_type', '?')})")
    if hasattr(record_set, 'columns') and record_set.columns:
        print("    Columns:")
        for column in record_set.columns:
            print(f"      - {column.name} (@id: {column.id}, type: {getattr(column, 'data_type', '?')})")

## 3. Data Extraction

Load data from all primary record sets into pandas DataFrames using the record set `@id`s.

In [ ]:
# Collect data from each primary record set using their @id.

# Use the ids from rs_list in the previous step.
import collections

dataframes = collections.OrderedDict()

for record_set_id in rs_list:
    # each record is a dictionary with field/column names as keys
    # (values may be None for missing data)
    record_set_records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(record_set_records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set @{record_set_id}.")

# For demonstration, show columns of the first record set
if rs_list:
    first_rs = rs_list[0]
    print(f"\nColumns in record set @{first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)

Apply some typical data processing and summarization, focusing on numeric and categorical fields.

Here: 
- Select a numeric field by its `@id`, 
- Filter for values above a threshold, 
- Normalize and group by a categorical field where appropriate.

Replace the `numeric_field_id` and `group_field_id` below with meaningful `@id` values from the printed record set fields above.

In [ ]:
# Example: Analyze the first record set
record_set_id = rs_list[0]  # or select an appropriate record set
df = dataframes[record_set_id]

# Select a numeric field's @id from the columns above (based on field definitions)
# E.g. suppose field for 'Age' has @id: 'https://api.app.sen.science/frontiers/7862866/field-age'
numeric_field_id = None
# Try to auto-detect common fields
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
        break
if not numeric_field_id and len(df.columns) > 0:
    numeric_field_id = df.columns[0]

print(f"Using numeric field for EDA: {numeric_field_id}")

threshold = 40  # e.g. age > 40
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold]
else:
    # Try convert if not numeric yet
    filtered_df = df.copy()
    filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Find a likely grouping/categorical field (e.g. Sex, cancer type, etc)
group_field_id = None
for col in df.columns:
    if (
        'sex' in col.lower() or 'gender' in col.lower()
        or 'site' in col.lower() or 'group' in col.lower()
        or 'location' in col.lower() or 'msi' in col.lower()
    ):
        group_field_id = col
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    display(grouped_df)

## 5. Visualization

Visualize data distributions and relationships between fields.

Below: Histogram for the selected numeric field, and a boxplot of that field grouped by a categorical variable (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot if grouping field is available
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

- This notebook demonstrated accessing and analyzing the FAIR$^2$ dataset using `mlcroissant`.
- All record sets, fields, and columns were referenced by their schema `@id`s for reproducibility.
- Basic inspection, filtering, normalization, grouping, and visualizations were performed, providing a foundation for further analysis and ML workflows.

Review all field `@id`s and data dictionary for precise medical field usage.

For in-depth analysis, consult clinical experts and the Croissant schema documentation.